In [1]:
!pip install datasets==3.5.0 fsspec==2024.12.0 gcsfs==2024.12.0
!pip install transformers hf_xet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 10.9 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: gcsfs
    Found existing installation: gcsfs 2025.3.2
    Uninstalling gcsfs-2025.3.2:
      Successfully uninstalled gcsfs-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine 

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel
from sklearn.metrics import accuracy_score, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import datasets

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load and preprocess dataset
def tokenize_batch(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

imdb_dataset = datasets.load_dataset('imdb')
tokenized_dataset = imdb_dataset.map(tokenize_batch, batched=True)

# Set dataset format
tokenized_dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'label']
)

# Data loaders
train_loader = DataLoader(tokenized_dataset['train'], batch_size=32, shuffle=True)
test_loader = DataLoader(tokenized_dataset['test'], batch_size=32)


Using device: cpu


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
print(f"Train dataset size: {len(tokenized_dataset['train'])}")
print(f"Test dataset size: {len(tokenized_dataset['test'])}")

Train dataset size: 25000
Test dataset size: 25000


In [4]:
print(f"Train sample: {tokenized_dataset['train'][0]}")
print(f"Test sample: {tokenized_dataset['test'][0]}")

Train sample: {'label': tensor(0), 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
         2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
         2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
         1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
         2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
         6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
         1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
         5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
        14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
         1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
         2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
        25430, 1

# label
What it Represents:
The label is a tensor that represents the ground-truth class for the sample. In this example, it is tensor(0).

Meaning:
In binary sentiment analysis (e.g., IMDB dataset), a label of 0 typically represents one class (commonly Negative), while 1 would represent the Positive class.

# input_ids
What it Represents:
The input_ids tensor contains the numerical token IDs obtained after the text is processed by the BERT tokenizer.

Special Tokens:

The first token, 101, represents the [CLS] token.

The token 102 at the end usually represents the [SEP] token, marking the end of the sequence.

Shape:
The input_ids are padded to a fixed length—specified as max_length=128 in the tokenization function. Therefore, its shape is (128,).
For example, if the batch size were larger, each sample in the batch would have a 1D tensor with 128 elements.

# attention_mask
What it Represents:
The attention_mask is a binary tensor that informs the model which tokens should be attended to. A 1 means the token is relevant (part of the actual input), and a 0 denotes padding.

Shape:
Similar to input_ids, this tensor is also of fixed length (128,), matching the sequence length after padding.

In [ ]:
# Define BERT classifier
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name='bert-base-uncased', num_classes=2):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# Initialize model, optimizer, and loss
model = BERTClassifier().to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
criterion = nn.CrossEntropyLoss()

# outputs (Model Output):

## last_hidden_state:

Shape: (batch_size, sequence_length, hidden_size)
For instance, with a batch size of 16, sequence length 128, and using bert-base-uncased with a hidden size of 768, the shape would be (16, 128, 768).

## pooler_output:

Shape: (batch_size, hidden_size)
In the same example, the shape would be (16, 768).

The BERT model processes the input_ids (using attention_mask to disregard padding) to produce the last_hidden_state which contains token-level representations. The pooler_output is a special pooled representation of the sequence (typically using the [CLS] token's representation after a linear layer and tanh activation) which is used for tasks like classification.

In [ ]:
# Training function
def train_epoch(model, data_loader, optimizer, criterion, device):
    model.train()
    total_loss, all_preds, all_labels = 0, [], []

    for batch in tqdm(data_loader, desc="Training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy

# Evaluation function
def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss, all_preds, all_labels = 0, [], []

    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            total_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_preds)
    report = classification_report(all_labels, all_preds, target_names=['Negative', 'Positive'])

    return avg_loss, accuracy, report

In [ ]:
# Training loop with best model saving
best_accuracy = 0
train_losses, train_accs, val_losses, val_accs = [], [], [], []

for epoch in range(3):
    print(f"Epoch {epoch+1}/3")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, report = evaluate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")
    print(report)

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), 'best_bert_imdb_classifier.pt')

# Plot results
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss per Epoch')

plt.subplot(1, 2, 2)
plt.plot(train_accs, label='Train Accuracy')
plt.plot(val_accs, label='Validation Accuracy')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy per Epoch')

plt.tight_layout()
plt.savefig('bert_imdb_training.png')
plt.show()

Epoch 1/5


Evaluating: 100%|██████████| 782/782 [02:58<00:00,  4.39it/s]


Train Loss: 0.3412, Train Accuracy: 0.8495
Validation Loss: 0.2836, Validation Accuracy: 0.8796
              precision    recall  f1-score   support

    Negative       0.92      0.83      0.87     12500
    Positive       0.84      0.93      0.89     12500

    accuracy                           0.88     25000
   macro avg       0.88      0.88      0.88     25000
weighted avg       0.88      0.88      0.88     25000

Epoch 2/5


Evaluating: 100%|██████████| 782/782 [02:58<00:00,  4.38it/s]


Train Loss: 0.2283, Train Accuracy: 0.9085
Validation Loss: 0.2677, Validation Accuracy: 0.8896
              precision    recall  f1-score   support

    Negative       0.90      0.88      0.89     12500
    Positive       0.88      0.90      0.89     12500

    accuracy                           0.89     25000
   macro avg       0.89      0.89      0.89     25000
weighted avg       0.89      0.89      0.89     25000

Epoch 3/5


Evaluating:  71%|███████▏  | 559/782 [02:07<00:51,  4.37it/s]